In [4]:
from azure.storage.blob import BlobServiceClient
import torch
from transformers import AutoModel, AutoTokenizer
import os
from datasets import Dataset
import pandas as pd
import numpy as np

# Azure Storage details
AZURE_STORAGE_CONNECTION_STRING = os.getenv("SONAR_STORAGE_KEY")
CONTAINER_NAME = "results"

# Create BlobServiceClient
blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# Local folder to store downloaded model
local_model_dir = "/Users/jonasklein/biasintransformers/downloaded_model"
os.makedirs(local_model_dir, exist_ok=True)

# Local folder to store downloaded dataset
local_dataset_dir = "/Users/jonasklein/biasintransformers/downloaded_dataset"
os.makedirs(local_dataset_dir, exist_ok=True)

### Loading in checkpoint model and train dataset

In [ ]:
MODEL_PATH = "bert_good_25/bert_checkpoints/checkpoint-201000"

# List of model files to download
model_files = [
    "config.json",
    "generation_config.json",
    "model.safetensors",
    "optimizer.pt",
    "rng_state.pth",
    "scheduler.pt",
    "special_tokens_map.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "trainer_state.json",
    "training_args.bin",
    "vocab.txt"
]

# Download model files
for file in model_files:
    blob_client = container_client.get_blob_client(f"{MODEL_PATH}/{file}")
    local_file_path = os.path.join(local_model_dir, file)

    with open(local_file_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())
    print(f"Downloaded {file} to {local_file_path}")

In [5]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_dir)
print(tokenizer)

BertTokenizerFast(name_or_path='/Users/jonasklein/biasintransformers/downloaded_model', vocab_size=30000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [ ]:
DATASET_PATH = "bert_good_25/train_dataset"

# List of dataset files to download
dataset_files = [
    "data-00000-of-00001.arrow",
    "dataset_info.json",
    "state.json"
]

# Download dataset files
for file in dataset_files:
    blob_client = container_client.get_blob_client(f"{DATASET_PATH}/{file}")
    local_file_path = os.path.join(local_dataset_dir, file)

    with open(local_file_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())
    print(f"Downloaded {file} to {local_file_path}")

BertTokenizerFast(name_or_path='/Users/jonasklein/biasintransformers/downloaded_model', vocab_size=30000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
Downloaded data-00000-of-00001.arrow to /Users/jonasklein/biasintransfo

### Checking the train dataset with the tokenizer

In [6]:
# Find the token ids of a certain word with the tokenizer
word = "man"
word_ids = tokenizer.encode(word, add_special_tokens=False)
print(f"Token ids of '{word}': {word_ids}")

# Decode these token ids back to a word
decoded_word = tokenizer.convert_ids_to_tokens(word_ids)
print(f"Decoded word: {decoded_word}")

Token ids of 'man': [497]
Decoded word: ['man']


In [7]:
# Load dataset from the Arrow file
dataset = Dataset.from_file(local_dataset_dir + "/data-00000-of-00001.arrow")

# Get the first example
first_example = dataset[0]

# Print the first example
print(first_example)

df = pd.DataFrame(dataset)
print(df.head())
# Print the size of the dataset
print(len(dataset))

# Decode the first example using the tokenizer, while keeping the tokens separated
decoded_example = tokenizer.convert_ids_to_tokens(first_example["input_ids"])
print(decoded_example)
print(len(decoded_example))

{'input_ids': [2, 2757, 711, 303, 364, 5899, 3766, 263, 235, 924, 1235, 304, 4189, 3229, 243, 1235, 19442, 434, 435, 1694, 292, 1054, 18, 18, 18, 323, 485, 303, 532, 295, 325, 11, 80, 16815, 500, 6500, 16, 1286, 235, 1788, 611, 871, 18, 851, 317, 618, 432, 718, 8275, 1417, 328, 235, 10293, 11, 85, 244, 245, 482, 16, 323, 2935, 317, 303, 760, 247, 2334, 2098, 2894, 18, 411, 317, 7436, 295, 775, 1574, 243, 368, 335, 459, 5895, 243, 2364, 448, 385, 718, 13834, 282, 313, 282, 1213, 380, 435, 1134, 317, 1417, 18, 5071, 317, 247, 7277, 244, 1345, 13712, 16, 323, 278, 500, 485, 392, 8601, 18, 352, 290, 6323, 3976, 297, 295, 18814, 6500, 258, 478, 18, 609, 290, 435, 8941, 292, 18, 1319, 711, 303, 1222, 881, 363, 6807, 290, 16, 235, 956, 4817, 244, 235, 8694, 18, 411, 588, 2657, 539, 258, 596, 297, 235, 4692, 258, 5232, 17, 17, 323, 363, 35, 29438, 878, 1390, 4865, 282, 1042, 18, 609, 734, 247, 4868, 282, 1007, 263, 18, 825, 303, 245, 4333, 244, 235, 5247, 6511, 17, 1573, 2920, 2991, 16, 290, 3

In [8]:
# Define the target word
word = "man"

# Get token IDs of the target word
word_ids = tokenizer.encode(word, add_special_tokens=False)
print(f"Token ids of '{word}': {word_ids}")

# Load dataset
dataset = Dataset.from_file(local_dataset_dir + "/data-00000-of-00001.arrow")

# Filter dataset: Keep only examples where input_ids contain word_ids
filtered_examples = [
    example for example in dataset 
    if any(
        example["input_ids"][i:i+len(word_ids)] == word_ids 
        for i in range(len(example["input_ids"]) - len(word_ids) + 1)
    )
]

filtered_df = pd.DataFrame(filtered_examples)

# Print dataset size before and after filtering
print(f"Original dataset size: {len(dataset)}")
print(f"Filtered dataset size: {len(filtered_df)}")

# Print the first few rows of the filtered dataset
print(filtered_df.head())

# Decode the first example using the tokenizer, while keeping the tokens separated
decoded_example = tokenizer.convert_ids_to_tokens(filtered_examples[0]["input_ids"])
print(decoded_example)
print(len(decoded_example))

Token ids of 'man': [497]
Original dataset size: 19968
Filtered dataset size: 3810
                                           input_ids
0  [2, 552, 87, 235, 4262, 244, 515, 2066, 835, 4...
1  [2, 333, 17053, 8425, 154, 16, 235, 26422, 16,...
2  [2, 333, 9855, 4088, 244, 245, 9465, 11653, 87...
3  [2, 320, 833, 632, 295, 247, 1919, 832, 286, 4...
4  [2, 1554, 278, 304, 767, 282, 16, 746, 316, 25...
['[CLS]', 'Als', 'u', 'de', 'leiding', 'van', 'dit', 'ma', '##mm', '##oet', '##pro', '##ject', 'over', 'de', 'Geschiedenis', 'van', 'het', 'Denken', 'in', 'de', 'Verenigde', 'Staten', 'niet', 'krijgt', ',', 'bent', 'u', 'als', 'docent', 'en', 'als', 'mens', 'niets', 'meer', 'waard', '.', 'Dan', 'rest', 'u', 'alleen', 'nog', 'maar', 'dat', 'slanke', 'lichaam', 'van', 'een', 'oude', 'tennis', '##ser', '.', 'Ik', 'houd', 'niet', 'van', 'de', 'sou', '##ple', '##ss', '##e', 'van', 'tennis', '##sers', ',', 'misschien', 'omdat', 'ik', 'zelf', 'altijd', 'dik', '##huid', '##ig', 'en', 'l', '##omp', 'b

In [9]:
# Find occurrences of word_ids in input_ids and store their indices
filtered_df[f"{word}_indices"] = [
    [i for i in range(len(example["input_ids"]) - len(word_ids) + 1)
     if example["input_ids"][i:i+len(word_ids)] == word_ids]
    for example in filtered_examples
]

# Print the first few rows of the filtered dataset
print(filtered_df.head())

                                           input_ids      man_indices
0  [2, 552, 87, 235, 4262, 244, 515, 2066, 835, 4...       [362, 485]
1  [2, 333, 17053, 8425, 154, 16, 235, 26422, 16,...            [327]
2  [2, 333, 9855, 4088, 244, 245, 9465, 11653, 87...            [309]
3  [2, 320, 833, 632, 295, 247, 1919, 832, 286, 4...             [91]
4  [2, 1554, 278, 304, 767, 282, 16, 746, 316, 25...  [188, 362, 437]


### Get the embedding for a certain sentence

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_dir)

# Load model
model = AutoModel.from_pretrained(local_model_dir)

model.eval()

# Test with an example input
text = "The doctor said that the patient had a fever. She recommended taking some medicine to reduce it."
inputs = tokenizer(text, return_tensors="pt")

# Print the tokenized input
print("Tokenized input:", inputs)

# Revert back the inputs variable to text, but showing the token id between brackets for each token
print("Tokenized input (with tokens):", tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze()))

# Perform forward pass to get hidden states
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

# Extract last hidden state (before MLM head)
last_hidden_state = outputs.hidden_states[-1]  # Shape: (batch_size, seq_length, hidden_dim)

print("Last hidden layer shape:", last_hidden_state.shape)

Some weights of BertModel were not initialized from the model checkpoint at /Users/jonasklein/biasintransformers/downloaded_model and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenized input: {'input_ids': tensor([[    2,  2494, 16255, 21887,   299,  2437,   238,   955,  2021, 15650,
           317,    67,   934,   298,    18,  8899,  5314,  4575,   246,   152,
          7696,   271,  5687,   138, 24956,   650,  4541, 23617,   138,    75,
           144,    18,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Tokenized input (with tokens): ['[CLS]', 'The', 'doctor', 'sa', '##id', 'th', '##at', 'the', 'pat', '##ient', 'had', 'a', 'fe', '##ver', '.', 'She', 'rec', '##ommen', '##de', '##d', 'tak', '##ing', 'som', '##e', 'medici', '##ne', 'to', 'reduc', '##e', 'i', '##t', '.', '[SEP]']
Last hidden layer shape: torch.Size([1, 33, 768])


In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(local_model_dir)
model = AutoModel.from_pretrained(local_model_dir)
model.eval()

# Dictionary to store embeddings for each row
embeddings_dict = {}

# Iterate over each row in the DataFrame
for index, row in filtered_df.iterrows():
    input_ids = row["input_ids"]
    word_indices = row[f"{word}_indices"]
    
    # Convert input_ids back to text
    text = tokenizer.decode(input_ids, skip_special_tokens=True)
    
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    
    # Get last hidden states
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    last_hidden_state = outputs.hidden_states[-1]
    
    # Extract embeddings for each word index
    row_embeddings = []
    for idx in word_indices:
        # Get the span of token indices for the word
        word_span = list(range(idx, idx + len(word_ids)))
        
        # Extract embeddings and average if multiple subwords
        word_embedding = last_hidden_state[:, word_span, :].mean(dim=1)
        
        row_embeddings.append(word_embedding.squeeze().numpy())
    
    # Store embeddings in dictionary
    embeddings_dict[index] = row_embeddings

# Convert dictionary to DataFrame for easier storage
embeddings_df = pd.DataFrame(list(embeddings_dict.items()), columns=["row_index", "embeddings"])

# Print first few rows
print(embeddings_df.head())

# Print the shape of the embeddings DataFrame
print(embeddings_df.shape)

# Save the embeddings DataFrame to a CSV file
embeddings_df.to_csv("embeddings.csv", index=False)

# Get the shape of the array in the first row, by first the list to a NumPy array
first_row_array = embeddings_df["embeddings"].iloc[1]
first_row_array = np.array(first_row_array)
print(first_row_array.shape)

Some weights of BertModel were not initialized from the model checkpoint at /Users/jonasklein/biasintransformers/downloaded_model and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   row_index                                         embeddings
0          0  [[-1.7398922, 0.4807269, 0.00029929407, 2.1737...
1          1  [[1.6417755, 0.17512035, -0.58163106, -0.02366...
2          2  [[-2.0371637, -0.36349893, -1.795675, 2.695723...
3          3  [[-1.6741303, -0.048178416, -0.7178874, 2.6838...
4          4  [[0.358163, -0.50406146, -1.3794116, 1.3129153...
(3810, 2)
(1, 768)


In [ ]:
# Initialize empty lists for embeddings and labels
all_embeddings = []
labels = []

# Iterate over DataFrame and add embeddings separately
for row in embeddings_df["embeddings"]:
    for embedding in row:  # Each row contains a list of embeddings
        all_embeddings.append(embedding)  # Append each embedding separately
        labels.append(0)  # Append label 0 for each embedding

# Convert to NumPy arrays
all_embeddings = np.array(all_embeddings)
labels = np.array(labels)

# Print shapes to verify
print("Total embeddings shape:", all_embeddings.shape)
print("Labels shape:", labels.shape)

# Save embeddings and labels to NumPy files
np.save("embeddings.npy", all_embeddings)
np.save("labels.npy", labels)

Total embeddings shape: (18, 768)
Labels shape: (18,)


In [ ]:
# Load embeddings and labels
loaded_embeddings = np.load("embeddings.npy")
loaded_labels = np.load("labels.npy")

# Print shapes to verify
print("Loaded embeddings shape:", loaded_embeddings.shape)
print("Loaded labels shape:", loaded_labels.shape)

# Access the first embedding and its label
print("First embedding:", loaded_embeddings[0])
print("First label:", loaded_labels[0])

Loaded embeddings shape: (18, 768)
Loaded labels shape: (18,)
First embedding: [-4.87418026e-01  3.95598114e-01 -4.86765385e-01  4.17527407e-01
  1.43280256e+00  1.19144833e+00  6.47811532e-01  1.19351745e+00
 -1.79303467e-01 -1.13604054e-01  3.24275076e-01  7.14560866e-01
  1.36532009e-01 -9.88516212e-01 -8.85097384e-01  2.00988844e-01
  1.11354005e+00 -3.12467843e-01 -3.03168297e-01  1.11985612e+00
  1.09883153e+00 -3.42380032e-02 -2.71268368e-01 -3.86933863e-01
  1.17793298e+00 -8.70060176e-02  1.43468595e+00 -1.18250287e+00
  1.25028920e+00  8.10745537e-01 -8.19120854e-02  6.08051777e-01
  3.88363063e-01  1.48587918e+00  4.03437167e-01 -1.17072821e+00
 -1.50258064e+00  2.61119366e-01 -1.66629270e-01  1.12658095e+00
 -4.80191678e-01 -1.84311497e+00 -7.76932240e-01 -5.18310308e-01
  6.51970804e-01 -2.48712301e-01  5.30876875e-01 -6.23175800e-01
  7.65490294e-01 -5.27769178e-02 -1.76890707e+00 -1.42070007e+00
  4.71376985e-01 -7.69802451e-01 -9.01436359e-02 -6.31482244e-01
  2.1074013